In [1]:
from typing import TypedDict, Any, Annotated
import operator
from pathlib import Path
import json
from langgraph.types import Send
import shutil
from langgraph.graph import StateGraph, START, END

In [2]:
class ModelingState(TypedDict, total=False):
    prepared_dataset_path: str
    current_dataset_path: str
    target_column: str
    task_type: str
    model_job: dict[str, Any]
    model_jobs: list[dict[str, Any]]
    model_results: Annotated[list[dict[str, Any]], operator.add]
    logs: Annotated[list[dict[str, Any]], operator.add]
    model_planning: dict[str, Any]
    model_comparison: dict[str, Any]
    tune_best_model: dict[str, Any]
    final_modeling: dict[str, Any]
    modeling_summary: dict[str, Any]
    artifacts: dict[str, Any]
    error: str | None
    status: str

In [3]:
def run_stage_with_llm_modeling(stage_name: str, task: str):
    print(f"\n========== START MODELING STAGE: {stage_name} ==========")
    try:
        result = modeling_stage_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        })
        raw = result["messages"][-1].content.strip()
        print("\nRAW RESPONSE:")
        print(raw)
        parsed = extract_json(raw)
        print(f"========== END MODELING STAGE: {stage_name} | status=success ==========")
        return parsed
    except Exception as e:
        print(f"========== END MODELING STAGE: {stage_name} | status=failed ==========")
        print(f"ERROR: {e}")
        raise

In [4]:
def modeling_plan_workers_node(state: ModelingState):
    prepared_dataset_path = state.get("prepared_dataset_path")
    target_column = state.get("target_column")
    task_type = state.get("task_type")

    if not prepared_dataset_path or not target_column or not task_type:
        error = (
            "Отсутствуют обязательные входные данные для этапа plan_model_workers: "
            f"prepared_dataset_path={prepared_dataset_path}, "
            f"target_column={target_column}, "
            f"task_type={task_type}"
        )

        new_log = {
            "agent": "modeling_agent",
            "stage": "plan_model_workers",
            "status": "failed",
            "summary": "Планирование model workers завершилось ошибкой, потому что отсутствуют обязательные входные данные",
            "reason": error,
        }

        return {
            "error": error,
            "logs": [new_log],
            "status": "failed",
        }

    task = f"""
Этап: plan_model_workers

Путь к подготовленному датасету:
{prepared_dataset_path}

Целевая переменная:
{target_column}

Тип задачи:
{task_type}

Используй python_interpreter_tool, если нужен компактный осмотр датасета

Твоя задача:
Сформировать план моделирования ровно с 2 независимыми model workers

Эти workers будут запущены параллельно на следующем этапе

Правила:
- Выбери ровно 2 модели
- Выбирай модели, подходящие под тип задачи
- Каждый worker должен обучать ровно одну модель
- На этом этапе не обучай модели
- Не изменяй датасет
- Верни только компактный JSON

Для регрессии предпочтительно использовать:
- Ridge как стабильную линейную baseline-модель
- RandomForestRegressor как нелинейную ансамблевую модель

Разрешенные модели для регрессии:
- LinearRegression
- Ridge
- Lasso
- RandomForestRegressor
- GradientBoostingRegressor

Верни только JSON:
{{
  "stage": "plan_model_workers",
  "status": "success",
  "result": {{
    "task_type": "{task_type}",
    "target_column": "{target_column}",
    "prepared_dataset_path": "{prepared_dataset_path}",
    "model_jobs": [
      {{
        "job_id": "model_1",
        "model_name": "Ridge",
        "model_type": "linear",
        "reason": "стабильная линейная baseline-модель"
      }},
      {{
        "job_id": "model_2",
        "model_name": "RandomForestRegressor",
        "model_type": "ensemble",
        "reason": "нелинейная ансамблевая модель"
      }}
    ],
    "primary_metric": "rmse",
    "selection_rule": "для регрессии ниже RMSE означает лучшее качество"
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_modeling("plan_model_workers", task)
        model_planning = parsed.get("result", {})
        model_jobs = model_planning.get("model_jobs", [])[:2]
        model_planning["model_jobs"] = model_jobs
        artifacts_dir = Path("artifacts/modeling/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)
        planning_path = artifacts_dir / "plan_model_workers.json"
        with open(planning_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "stage": "plan_model_workers",
                    "status": "success",
                    "result": model_planning,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )
        new_log = {
            "agent": "modeling_agent",
            "stage": "plan_model_workers",
            "status": "success",
            "summary": f"Planned {len(model_jobs)} model workers.",
            "artifacts": {
                "model_planning_path": str(planning_path),
            },
        }
        return {
            "model_planning": model_planning,
            "model_jobs": model_jobs,
            "model_results": [],
            "artifacts": {
                **state.get("artifacts", {}),
                "model_planning_path": str(planning_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }
    except Exception as e:
        new_log = {
            "agent": "modeling_agent",
            "stage": "plan_model_workers",
            "status": "failed",
            "summary": "Не удалось запланировать model workers",
            "reason": str(e),
        }
        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [5]:
def send_model_jobs(state: ModelingState):
    model_jobs = state.get("model_jobs", [])
    print(f"=== SEND MODEL JOBS | count={len(model_jobs)} ===")
    for job in model_jobs:
        print(f"Send model job: {job.get('job_id')} | {job.get('model_name')}")
    return [
        Send(
            "train_model_worker",
            {
                "prepared_dataset_path": state.get("prepared_dataset_path"),
                "current_dataset_path": state.get("current_dataset_path"),
                "target_column": state.get("target_column"),
                "task_type": state.get("task_type"),
                "model_job": job,
                "model_results": [],
                "logs": [],
            },
        )
        for job in model_jobs
    ]

In [6]:
def modeling_train_model_worker_node(state: ModelingState):
    prepared_dataset_path = state.get("prepared_dataset_path")
    target_column = state.get("target_column")
    task_type = state.get("task_type")
    model_job = state.get("model_job", {})

    if not prepared_dataset_path or not target_column or not task_type or not model_job:
        error = (
            "Отсутствуют обязательные входные данные для этапа train_model_worker: "
            f"prepared_dataset_path={prepared_dataset_path}, "
            f"target_column={target_column}, "
            f"task_type={task_type}, "
            f"model_job_empty={not bool(model_job)}"
        )

        failed_result = {
            "job_id": model_job.get("job_id"),
            "model_name": model_job.get("model_name"),
            "task_type": task_type,
            "status": "failed",
            "error": error,
            "metrics": {},
            "model_artifact_path": None,
            "metrics_path": None,
        }

        new_log = {
            "agent": "modeling_agent",
            "stage": "train_model_worker",
            "status": "failed",
            "summary": "Обучение model worker завершилось ошибкой, потому что отсутствуют обязательные входные данные",
            "reason": error,
        }

        return {
            "model_results": [failed_result],
            "logs": [new_log],
        }

    job_id = model_job["job_id"]
    model_name = model_job["model_name"]

    model_artifact_path = f"artifacts/modeling/models/{job_id}_{model_name}.joblib"
    metrics_path = f"artifacts/modeling/metrics/{job_id}_{model_name}_metrics.json"

    task = f"""
Этап: train_model_worker

Это один параллельный model worker

Путь к подготовленному датасету:
{prepared_dataset_path}

Целевая переменная:
{target_column}

Тип задачи:
{task_type}

Задание для модели:
{json.dumps(model_job, ensure_ascii=False, indent=2)}

Путь для сохранения обученной модели:
{model_artifact_path}

Путь для сохранения метрик:
{metrics_path}

Используй python_interpreter_tool, чтобы написать и выполнить короткий код на Python

Твоя задача:
Обучить ровно одну модель, указанную в model_job

Правила:
- Прочитай подготовленный датасет
- Раздели данные на train и test
- Обучение производи только на train
- Используй test_size=0.2 и random_state=42
- Не используй целевую переменную как признак
- Обучай только запрошенную модель
- Для регрессии рассчитай:
  mae, rmse, r2
- Названия метрик должны быть в нижнем регистре
- Сохрани обученную модель по указанному пути model_artifact_path с помощью joblib
- Сохрани JSON с метриками по указанному пути metrics_path
- JSON-файл с метриками должен иметь следующую структуру:
  {{
    "metrics": {{}},
    "n_train": 0,
    "n_test": 0
  }}
- Не изменяй подготовленный датасет
- Не обучай другие модели
- Верни только компактный JSON

Критически важные правила против галлюцинаций:
- Финальный JSON должен использовать ровно те метрики, которые были рассчитаны выполненным Python-кодом
- Не выдумывай и не приближай значения метрик
- Не выдумывай n_train и n_test
- После выполнения кода прочитай JSON-файл с метриками и используй эти значения в финальном ответе
- Значения в финальном JSON должны совпадать с сохраненным файлом метрик

Используй базовые настройки моделей:

Регрессия:
- LinearRegression: без дополнительных гиперпараметров
- Ridge: без дополнительных гиперпараметров
- Lasso: без дополнительных гиперпараметров
- RandomForestRegressor: без дополнительных гиперпараметров, кроме random_state=42
- GradientBoostingRegressor: без дополнительных гиперпараметров, кроме random_state=42
- CatBoostRegressor: использовать базовые настройки, но обязательно указать verbose=0 и random_seed=42

Дополнительные правила:
- Не запускай GridSearchCV или RandomizedSearchCV на этом этапе
- Не обучай модели, которые не указаны в model_job
- Для всех моделей, где доступен random_state, используй random_state=42
- Для CatBoostRegressor используй random_seed=42, а не random_state
- Если используется CatBoostRegressor, импортируй его так: from catboost import CatBoostRegressor

Поддерживаемое соответствие названий моделей и классов:

Регрессия:
- LinearRegression
- Ridge
- Lasso
- RandomForestRegressor
- GradientBoostingRegressor
- CatBoostRegressor

Верни только JSON:
{{
  "stage": "train_model_worker",
  "status": "success",
  "result": {{
    "job_id": "{job_id}",
    "model_name": "{model_name}",
    "task_type": "{task_type}",
    "model_artifact_path": "{model_artifact_path}",
    "metrics_path": "{metrics_path}",
    "metrics": {{}},
    "n_train": null,
    "n_test": null,
    "warnings": []
  }}
}}
"""

    try:
        print(f"=== WORKER STARTED: {job_id} | {model_name} ===")
        parsed = run_stage_with_llm_modeling(
            f"train_model_worker_{job_id}",
            task,
        )
        print(f"=== WORKER LLM FINISHED: {job_id} | {model_name} ===")
        result = parsed.get("result", {})
        result["metrics"] = parsed.get("metrics", result.get("metrics", {}))
        result["n_train"] = parsed.get("n_train", result.get("n_train"))
        result["n_test"] = parsed.get("n_test", result.get("n_test"))

        result["job_id"] = job_id
        result["model_name"] = model_name
        result["task_type"] = task_type
        result["model_artifact_path"] = model_artifact_path
        result["metrics_path"] = metrics_path
        result.setdefault("status", "success")
        result.setdefault("warnings", [])

        print(f"=== WORKER FINISHED: {job_id} | {model_name} ===")
        print("Worker metrics:", result.get("metrics"))
        print("Worker n_train:", result.get("n_train"))
        print("Worker n_test:", result.get("n_test"))

        new_log = {
            "agent": "modeling_agent",
            "stage": "train_model_worker",
            "status": "success",
            "summary": f"Model worker completed: {model_name}.",
            "artifacts": {
                "model_artifact_path": model_artifact_path,
                "metrics_path": metrics_path,
            },
        }

        return {
            "model_results": [result],
            "logs": [new_log],
        }

    except Exception as e:
        print(f"=== WORKER FAILED: {job_id} | {model_name} ===")
        print("Worker error:", str(e))

        failed_result = {
            "job_id": job_id,
            "model_name": model_name,
            "task_type": task_type,
            "status": "failed",
            "error": str(e),
            "metrics": {},
            "model_artifact_path": None,
            "metrics_path": None,
        }

        new_log = {
            "agent": "modeling_agent",
            "stage": "train_model_worker",
            "status": "failed",
            "summary": f"Model worker failed: {model_name}.",
            "reason": str(e),
        }

        return {
            "model_results": [failed_result],
            "logs": [new_log],
        }

In [7]:
def modeling_compare_models_node(state: ModelingState):
    model_results = state.get("model_results", [])
    task_type = state.get("task_type")
    artifacts_dir = Path("artifacts/modeling/stages")
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    comparison_path = artifacts_dir / "compare_models.json"

    successful_results = [
        r for r in model_results
        if r.get("metrics") and r.get("model_artifact_path")
    ]

    if not successful_results:
        comparison_result = {
            "best_job_id": None,
            "best_model_name": None,
            "best_model_path": None,
            "best_metrics": {},
            "primary_metric": None,
            "reason": "Нет успешно обученных моделей.",
            "all_results": model_results,
        }

        comparison = {
            "stage": "compare_models",
            "status": "failed",
            "result": comparison_result,
        }

        with open(comparison_path, "w", encoding="utf-8") as f:
            json.dump(comparison, f, ensure_ascii=False, indent=2)

        new_log = {
            "agent": "modeling_agent",
            "stage": "compare_models",
            "status": "failed",
            "summary": "Не удалось выбрать лучшую модель, потому что нет успешно обученных моделей",
            "artifacts": {
                "model_comparison_path": str(comparison_path),
            },
            "reason": "Нет успешно обученных моделей",
        }

        return {
            "model_comparison": comparison_result,
            "error": "Нет успешно обученных моделей",
            "artifacts": {
                **state.get("artifacts", {}),
                "model_comparison_path": str(comparison_path),
            },
            "logs": [new_log],
            "status": "failed",
        }

    best = min(
        successful_results,
        key=lambda r: r.get("metrics", {}).get("rmse", float("inf"))
    )
    primary_metric = "rmse"

    comparison_result = {
        "best_job_id": best.get("job_id"),
        "best_model_name": best.get("model_name"),
        "best_model_path": best.get("model_artifact_path"),
        "best_metrics": best.get("metrics"),
        "primary_metric": primary_metric,
        "all_results": model_results,
    }

    comparison = {
        "stage": "compare_models",
        "status": "success",
        "result": comparison_result,
    }

    with open(comparison_path, "w", encoding="utf-8") as f:
        json.dump(comparison, f, ensure_ascii=False, indent=2)

    new_log = {
        "agent": "modeling_agent",
        "stage": "compare_models",
        "status": "success",
        "summary": f"Выбрана лучшая модель: {best.get('model_name')}.",
        "artifacts": {
            "model_comparison_path": str(comparison_path),
            "best_model_path": best.get("model_artifact_path"),
        },
    }

    return {
        "model_comparison": comparison_result,
        "artifacts": {
            **state.get("artifacts", {}),
            "model_comparison_path": str(comparison_path),
            "best_model_path": best.get("model_artifact_path"),
        },
        "logs": [new_log],
        "status": "running",
        "error": None,
    }

In [8]:
def modeling_tune_best_model_node(state: ModelingState):
    prepared_dataset_path = state.get("prepared_dataset_path")
    target_column = state.get("target_column")
    task_type = state.get("task_type")

    comparison = state.get("model_comparison", {})
    best_model_name = comparison.get("best_model_name")

    if not prepared_dataset_path or not target_column or not task_type or not best_model_name:
        error = (
            "Отсутствуют обязательные входные данные для этапа tune_best_model: "
            f"prepared_dataset_path={prepared_dataset_path}, "
            f"target_column={target_column}, "
            f"task_type={task_type}, "
            f"best_model_name={best_model_name}"
        )

        new_log = {
            "agent": "modeling_agent",
            "stage": "tune_best_model",
            "status": "failed",
            "summary": "Подбор гиперпараметров завершился ошибкой, потому что отсутствуют обязательные входные данные",
            "reason": error,
        }

        return {
            "error": error,
            "logs": [new_log],
            "status": "failed",
        }

    tuned_model_path = f"artifacts/modeling/tuned_models/tuned_{best_model_name}.joblib"
    tuned_metrics_path = f"artifacts/modeling/tuned_models/tuned_{best_model_name}_metrics.json"

    task = f"""
Этап: tune_best_model

Путь к подготовленному датасету:
{prepared_dataset_path}

Целевая переменная:
{target_column}

Тип задачи:
{task_type}

Лучшая модель с предыдущего этапа сравнения:
{json.dumps(comparison, ensure_ascii=False, indent=2)}

Путь для сохранения настроенной модели:
{tuned_model_path}

Путь для сохранения метрик настроенной модели:
{tuned_metrics_path}

Используй python_interpreter_tool, чтобы написать и выполнить короткий код на Python

Твоя задача:
Подобрать гиперпараметры только для лучшей модели, выбранной на предыдущем этапе

Правила:
- Прочитай подготовленный датасет
- Раздели данные на train и test с test_size=0.2 и random_state=42
- Не используй целевую переменную как признак
- Настраивай только тип лучшей модели
- Используй GridSearchCV или RandomizedSearchCV
- Используй 3-fold cross-validation
- Используй легкие сетки параметров, чтобы выполнение не было слишком долгим
- Не выполняй подготовку данных
- Не обучай другие типы моделей
- Сохрани лучший настроенный estimator по указанному пути tuned_model_path с помощью joblib
- Сохрани метрики и best_params по указанному пути tuned_metrics_path
- JSON-файл с метриками настроенной модели должен иметь следующую структуру:
  {{
    "best_params": {{}},
    "metrics": {{}},
    "n_train": 0,
    "n_test": 0
  }}
- Верни только компактный JSON

Настройка для регрессии:
- Для Ridge подбирай alpha
- Для Lasso подбирай alpha
- Для RandomForestRegressor подбирай n_estimators, max_depth, min_samples_split
- Для GradientBoostingRegressor подбирай n_estimators, learning_rate, max_depth
- Для CatBoostRegressor подбирай iterations, depth, learning_rate
- Для CatBoostRegressor обязательно используй verbose=0 и random_seed=42
- Метрика для scoring: neg_root_mean_squared_error
- Финальные test-метрики: mae, rmse, r2


Критически важные правила против галлюцинаций:
- Финальный JSON должен использовать ровно те метрики, которые были рассчитаны выполненным Python-кодом
- Не выдумывай и не приближай значения метрик
- Не выдумывай best_params
- Не выдумывай n_train и n_test
- После выполнения кода прочитай JSON-файл с метриками настроенной модели и используй эти значения в финальном ответе
- Значения в финальном JSON должны совпадать с сохраненным файлом метрик
- Названия метрик должны быть в нижнем регистре

Используй эти легкие сетки параметров:

Регрессия:
- Ridge: {{"alpha": [0.1, 1.0, 10.0, 50.0]}}
- Lasso: {{"alpha": [0.0001, 0.001, 0.01], "max_iter": [3000]}}
- RandomForestRegressor: {{"n_estimators": [50, 100], "max_depth": [8, 12, None], "min_samples_split": [2, 5]}}
- GradientBoostingRegressor: {{"n_estimators": [50, 100], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}}
- CatBoostRegressor: {{"iterations": [100, 200], "depth": [4, 6], "learning_rate": [0.03, 0.1], "verbose": [0], "random_seed": [42]}}

Верни только JSON:
{{
  "stage": "tune_best_model",
  "status": "success",
  "result": {{
    "best_model_name": "{best_model_name}",
    "tuned_model_path": "{tuned_model_path}",
    "tuned_metrics_path": "{tuned_metrics_path}",
    "best_params": {{}},
    "metrics": {{}},
    "n_train": null,
    "n_test": null,
    "warnings": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm_modeling("tune_best_model", task)
        tune_result = parsed.get("result", {})

        tune_result["best_model_name"] = best_model_name
        tune_result["tuned_model_path"] = tuned_model_path
        tune_result["tuned_metrics_path"] = tuned_metrics_path

        artifacts_dir = Path("artifacts/modeling/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        tune_stage_path = artifacts_dir / "tune_best_model.json"

        with open(tune_stage_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "stage": "tune_best_model",
                    "status": "success",
                    "result": tune_result,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        new_log = {
            "agent": "modeling_agent",
            "stage": "tune_best_model",
            "status": "success",
            "summary": f"Подбор гиперпараметров для модели {best_model_name} успешно завершен",
            "artifacts": {
                "tuned_model_path": tuned_model_path,
                "tuned_metrics_path": tuned_metrics_path,
                "tune_best_model_path": str(tune_stage_path),
            },
        }

        return {
            "tune_best_model": tune_result,
            "artifacts": {
                **state.get("artifacts", {}),
                "tuned_model_path": tuned_model_path,
                "tuned_metrics_path": tuned_metrics_path,
                "tune_best_model_path": str(tune_stage_path),
            },
            "logs": [new_log],
            "status": "running",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "modeling_agent",
            "stage": "tune_best_model",
            "status": "failed",
            "summary": "Подбор гиперпараметров завершился ошибкой",
            "reason": str(e),
        }

        return {
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }

In [9]:
def modeling_final_node(state: ModelingState):
    comparison = state.get("model_comparison", {})
    tuning = state.get("tune_best_model", {})

    best_model_name = comparison.get("best_model_name")

    best_model_path = (
        tuning.get("tuned_model_path")
        or comparison.get("best_model_path")
    )

    best_metrics = (
        tuning.get("metrics")
        or comparison.get("best_metrics", {})
    )

    best_params = tuning.get("best_params", {})

    final_model_path = "artifacts/modeling/best_model.joblib"
    final_metrics_path = "artifacts/modeling/final_metrics.json"
    modeling_summary_path = "artifacts/modeling/modeling_summary.json"

    Path("artifacts/modeling").mkdir(parents=True, exist_ok=True)

    if not best_model_path:
        new_log = {
            "agent": "modeling_agent",
            "stage": "final_modeling",
            "status": "failed",
            "summary": "Финальный результат моделирования завершилась ошибкой, потому что отсутствует путь к лучшей модели",
            "reason": "best_model_path отсутствует.",
        }

        return {
            "error": "best_model_path отсутствует",
            "logs": [new_log],
            "status": "failed",
        }

    shutil.copy(best_model_path, final_model_path)

    modeling_summary = {
        "status": "success",
        "task_type": state.get("task_type"),
        "target_column": state.get("target_column"),
        "prepared_dataset_path": state.get("prepared_dataset_path"),
        "models_tested": [
            r.get("model_name")
            for r in state.get("model_results", [])
            if r.get("model_name")
        ],
        "best_model_name": best_model_name,
        "best_model_path": final_model_path,
        "best_metrics": best_metrics,
        "tuning_used": bool(tuning),
        "best_params": best_params,
        "baseline_best_metrics": comparison.get("best_metrics", {}),
        "tuned_metrics": tuning.get("metrics", {}),
        "model_comparison": comparison,
        "tuning_result": tuning,
    }

    with open(final_metrics_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "best_model_name": best_model_name,
                "best_metrics": best_metrics,
                "best_params": best_params,
                "tuning_used": bool(tuning),
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    with open(modeling_summary_path, "w", encoding="utf-8") as f:
        json.dump(modeling_summary, f, ensure_ascii=False, indent=2)

    new_log = {
        "agent": "modeling_agent",
        "stage": "final_modeling",
        "status": "success",
        "summary": f"Финальная лучшая модель сохранена: {best_model_name}",
        "artifacts": {
            "best_model_path": final_model_path,
            "final_metrics_path": final_metrics_path,
            "modeling_summary_path": modeling_summary_path,
        },
    }

    return {
        "final_modeling": {
            "best_model_name": best_model_name,
            "best_model_path": final_model_path,
            "final_metrics_path": final_metrics_path,
            "modeling_summary_path": modeling_summary_path,
            "tuning_used": bool(tuning),
            "best_params": best_params,
        },
        "modeling_summary": modeling_summary,
        "artifacts": {
            **state.get("artifacts", {}),
            "best_model_path": final_model_path,
            "final_metrics_path": final_metrics_path,
            "modeling_summary_path": modeling_summary_path,
        },
        "logs": [new_log],
        "status": "success",
        "error": None,
    }

In [10]:
def modeling_join_workers_node(state: ModelingState):
    print("=== JOIN MODEL WORKERS ===")
    print("Model results count:", len(state.get("model_results", [])))

    return {
        "status": "running",
        "error": None,
    }


def build_modeling_graph():
    print("=== BUILD MODELING GRAPH ===")

    modeling_graph = StateGraph(ModelingState)

    modeling_graph.add_node("plan_model_workers", modeling_plan_workers_node)
    modeling_graph.add_node("train_model_worker", modeling_train_model_worker_node)
    modeling_graph.add_node("join_model_workers", modeling_join_workers_node)
    modeling_graph.add_node("compare_models", modeling_compare_models_node)
    modeling_graph.add_node("tune_best_model", modeling_tune_best_model_node)
    modeling_graph.add_node("final_modeling", modeling_final_node)

    modeling_graph.add_edge(START, "plan_model_workers")

    modeling_graph.add_conditional_edges(
        "plan_model_workers",
        send_model_jobs,
        ["train_model_worker"],
    )

    modeling_graph.add_edge("train_model_worker", "join_model_workers")
    modeling_graph.add_edge("join_model_workers", "compare_models")
    modeling_graph.add_edge("compare_models", "tune_best_model")
    modeling_graph.add_edge("tune_best_model", "final_modeling")
    modeling_graph.add_edge("final_modeling", END)

    return modeling_graph.compile()

In [11]:
def run_modeling_agent_node(state: AgentState):
    print("=== START MODELING AGENT ===")

    prepared_dataset_path = (
        state.get("prepared_dataset_path")
        or state.get("artifacts", {}).get("prepared_dataset_path")
        or state.get("next_input", {}).get("prepared_dataset_path")
    )

    target_column = (
        state.get("target_column")
        or state.get("next_input", {}).get("target_column")
    )

    task_type = (
        state.get("task_type")
        or state.get("next_input", {}).get("task_type")
    )

    print("Входные данные для Modeling Agent:")
    print("prepared_dataset_path:", prepared_dataset_path)
    print("target_column:", target_column)
    print("task_type:", task_type)

    if not prepared_dataset_path or not target_column or not task_type:
        error = (
            "Входные данные для Modeling Agent неполные: "
            f"prepared_dataset_path={prepared_dataset_path}, "
            f"target_column={target_column}, "
            f"task_type={task_type}"
        )

        print("Ошибка входных данных Modeling Agent:", error)

        log_record = {
            "agent": "modeling_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Входные данные для Modeling Agent неполные",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": error,
        }

        return {
            "current_step": "modeling_agent",
            "status": "failed",
            "error": error,
            "logs": [log_record],
        }

    try:
        print("Сборка modeling graph")
        modeling_app = build_modeling_graph()
        print("Modeling graph собран")

        modeling_state: ModelingState = {
            "prepared_dataset_path": prepared_dataset_path,
            "current_dataset_path": prepared_dataset_path,
            "target_column": target_column,
            "task_type": task_type,
            "model_jobs": [],
            "model_results": [],
            "model_planning": {},
            "model_comparison": {},
            "tune_best_model": {},
            "final_modeling": {},
            "modeling_summary": {},
            "artifacts": {},
            "logs": [],
            "status": "running",
            "error": None,
        }

        print("Запуск modeling graph")
        final_modeling_state = modeling_app.invoke(
            modeling_state,
            {"recursion_limit": 80},
        )
        print("Modeling graph завершил работу")
        print("Финальный статус Modeling Agent:", final_modeling_state.get("status"))
        print("Финальная ошибка Modeling Agent:", final_modeling_state.get("error"))

        merged_artifacts = {
            **state.get("artifacts", {}),
            **final_modeling_state.get("artifacts", {}),
        }

        modeling_summary = final_modeling_state.get("modeling_summary", {})

        reporting_next_input = {
            "target_column": target_column,
            "task_type": task_type,
            "prepared_dataset_path": prepared_dataset_path,
            "best_model_path": merged_artifacts.get("best_model_path"),
            "final_metrics_path": merged_artifacts.get("final_metrics_path"),
            "modeling_summary_path": merged_artifacts.get("modeling_summary_path"),
        }

        if final_modeling_state.get("status") != "success":
            error = final_modeling_state.get(
                "error",
                "Modeling Agent завершился ошибкой"
            )

            log_record = {
                "agent": "modeling_agent",
                "status": "failed",
                "skipped": False,
                "summary": "Modeling Agent завершился ошибкой",
                "decisions": {
                    "best_model_name": modeling_summary.get("best_model_name"),
                    "best_metrics": modeling_summary.get("best_metrics"),
                    "best_params": modeling_summary.get("best_params"),
                },
                "artifacts": final_modeling_state.get("artifacts", {}),
                "next_input": {},
                "reason": error,
                "subgraph_logs": final_modeling_state.get("logs", []),
            }

            return {
                "current_step": "modeling_agent",
                "status": "failed",
                "error": error,
                "artifacts": merged_artifacts,
                "logs": [log_record],
            }

        log_record = {
            "agent": "modeling_agent",
            "status": "success",
            "skipped": False,
            "summary": "Modeling Agent успешно завершил работу",
            "decisions": {
                "best_model_name": modeling_summary.get("best_model_name"),
                "best_metrics": modeling_summary.get("best_metrics"),
                "best_params": modeling_summary.get("best_params"),
                "tuning_used": modeling_summary.get("tuning_used"),
            },
            "artifacts": final_modeling_state.get("artifacts", {}),
            "next_input": reporting_next_input,
            "reason": None,
            "subgraph_logs": final_modeling_state.get("logs", []),
        }

        return {
            "current_step": "modeling_agent",
            "modeling_summary": modeling_summary,
            "next_input": reporting_next_input,
            "artifacts": merged_artifacts,
            "logs": [log_record],
            "next_action": None,
            "orchestration_decision": {},
            "status": "running",
            "error": None,
        }

    except Exception as e:
        print("Исключение в wrapper Modeling Agent:", str(e))

        log_record = {
            "agent": "modeling_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Modeling Agent завершился ошибкой",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": str(e),
        }

        return {
            "current_step": "modeling_agent",
            "status": "failed",
            "error": str(e),
            "logs": [log_record],
        }

NameError: name 'AgentState' is not defined